# Tutoriel

In [ ]:
import numpy as np
import pandas as pd

## Lire des données d'observation

### Au format XML-SANDRE (export HydroPortail)

In [ ]:
from evalhyd.vigicrues.read import read_obs_from_xml_sandre

In [ ]:
df_obs = read_obs_from_xml_sandre(['...'])

## Lire des données de prévision

### Au format XML-SANDRE

In [ ]:
from evalhyd.vigicrues.read import read_prd_from_xml_sandre

In [ ]:
df_prd = read_prd_from_xml_sandre(['...'])

### Au format PRV

In [ ]:
from evalhyd.vigicrues.read import read_prd_from_prv

In [ ]:
df_prd = read_prd_from_prv(['...'])

## Évaluer des prévisions

### Prévisions probabilistes

In [ ]:
from evalhyd.vigicrues.evaluate import evalp

#### Prévisions en scénarios

Tous les indicateurs sont calculables.

In [ ]:
dict_df_prb = evalp(
    df_obs=df_obs,
    df_prd=df_prd,
    metrics=['CRPS_FROM_ECDF', 'QS', 'CR', 'WS', 'REL_DIAG', 'RANK_HIST'],
    c_lvl=np.array([80])
)

In [ ]:
df_crps = dict_df_prb['CRPS_FROM_ECDF']

#### Prévisions en tendances

Seuls certains indicateurs sont calculables.

In [ ]:
dict_df_prb = evalp(
    df_obs=df_obs,
    df_prd=df_prd,
    metrics=['QS', 'CR', 'AW', 'AWN', 'WS'],
    c_lvl=np.array([80])
)

In [ ]:
df_qs = dict_df_prb['QS']

### Prévisions déterministes

In [ ]:
from evalhyd.vigicrues.evaluate import evald

Il est nécessaire de travailler entité par entité pour le calcul d'indicateurs déterministes.

In [ ]:
df_kge = None
df_cont_tbl = None

In [ ]:
for entite in df_obs.index.levels[0]:
    dict_df_dtm = evald(
        df_obs=df_obs.xs(entite, level='entites'),
        df_prd=df_prd.xs(entite, level='entites').xs('moy', level='membres',),
        metrics=['KGE', 'CONT_TBL']
    )

    df_kge = pd.concat(
        [df_kge, pd.concat({entite: dict_df_dtm['KGE']}, names=['entites'])]
    )
    df_cont_tbl = pd.concat(
        [df_cont_tbl, pd.concat({entite: dict_df_dtm['CONT_TBL']}, names=['entites'])]
    )

## Visualiser des indicateurs de prévision

### Histogramme de rang

In [ ]:
from evalhyd.vigicrues.plot import plot_rank_hist

In [ ]:
filepaths_rank_hist = plot_rank_hist(dict_df_prb['RANK_HIST'])

In [ ]:
from IPython.display import Image
Image(filepaths_rank_hist[0])

### Diagramme de fiabilité

In [ ]:
from evalhyd.vigicrues.plot import plot_rel_diag

In [ ]:
filepaths_rel_diag = plot_rel_diag(dict_df_prb['REL_DIAG'])

In [ ]:
from IPython.display import Image
Image(filepaths_rel_diag[0])